# 03 · QC filtering

*sc-rnaseq-wf, re-instantiated step by step for `team-jakobsson` (`jakobsson-pmdbs-sn-rnaseq`)*

| | |
| :- | :- |
| WDL task | [`filter`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/workflows/cohort_analysis/cohort_analysis.wdl) |
| Script | [`filter`](https://github.com/ASAP-CRN/sc-rnaseq-wf/blob/main/docker/sc_tools/scripts/main/filter) → `sc-filter` |

Each cell is kept only if it passes all four filters (workflow defaults):

| Metric | Keep if | Removes |
| :- | :- | :- |
| `pct_counts_mt` | ≤ 10 | damaged nuclei / cytoplasmic contamination |
| `doublet_score` | < 0.2 | likely doublets (scrublet) |
| `total_counts` | 500 – 150,000, and within 3 MADs below / 5 MADs above the sample median | empty droplets / debris, and outliers for the sample |
| `n_genes_by_counts` | 300 – 15,000, and within 3 MADs below / 5 MADs above the sample median | low-complexity droplets, and outliers for the sample |

**Median absolute deviation (MAD) filters.** Sequencing depth varies between samples, so fixed limits over-filter shallow samples and under-filter deep ones. The MAD thresholds are computed **within each sample** on the log1p scale (counts are right-skewed). The upper bound is looser than the lower one because high counts are usually fine, so only the extreme high tail is trimmed. If a sample's MAD is 0, only the absolute limits apply.

The object is opened in backed mode, the filters are evaluated on `obs`, and only the passing cells are read into memory. Every removed cell is written to `filtered_out_cells.csv` with one `failed_*` column per filter, so removed cells can be checked later (e.g. for an over-represented cell type).

**Inputs**
- `${cohort_id}.merged_cleaned_unfiltered.h5ad`

**Outputs** (in `cfg.RUN_DIR`, named as in the WDL)
- `${cohort_id}.filtered.h5ad`
- `${cohort_id}.filtered_out_cells.csv`: every removed cell and the filters it failed

**Previous:** [02_merge_and_plot_qc](02_merge_and_plot_qc.ipynb) · **Next:** [04_mmc](04_mmc.ipynb)

In [ ]:
import wf_config as cfg
import pandas as pd
import scanpy as sc
from IPython.display import Image, display

cfg.RUN_DIR.mkdir(parents=True, exist_ok=True)
print("run directory:", cfg.RUN_DIR)

## 1. Preview the filters on the metadata

The same conditions, evaluated on `initial_metadata.csv`, show how many cells each filter removes (a cell can fail several).

In [ ]:
import numpy as np
from scipy.stats import median_abs_deviation

obs = pd.read_csv(cfg.out("initial_metadata.csv"), index_col=0)

failed = pd.DataFrame({
    "pct_counts_mt": obs["pct_counts_mt"] > cfg.PCT_COUNTS_MT_MAX,
    "doublet_score": obs["doublet_score"] >= cfg.DOUBLET_SCORE_MAX,
    "total_counts": ~obs["total_counts"].between(*cfg.TOTAL_COUNTS_LIMITS),
    "n_genes_by_counts": ~obs["n_genes_by_counts"].between(*cfg.N_GENES_BY_COUNTS_LIMITS),
})

# Per-sample MAD outliers on the log1p scale, as in filter.mad_outlier
def mad_outliers(values: pd.Series) -> pd.Series:
    logged = np.log1p(values)
    median, mad = logged.median(), median_abs_deviation(logged)
    if mad == 0:
        return pd.Series(False, index=values.index)
    return (logged < median - cfg.N_MADS_LOWER * mad) | (logged > median + cfg.N_MADS_UPPER * mad)

for metric in ["total_counts", "n_genes_by_counts"]:
    failed[f"{metric}_mad"] = obs.groupby("sample", observed=True)[metric].transform(mad_outliers).astype(bool)

print(f"cells: {len(obs):,}; removed by any filter: {failed.any(axis=1).sum():,}")
failed.sum().rename("cells failing").to_frame()

In [ ]:
failed.groupby(obs["sample"]).mean().mul(100).round(1).add_suffix(" %")

## 2. Run `sc-filter`

In [ ]:
cfg.run(
    "sc-filter",
    "--adata-input", cfg.out("merged_cleaned_unfiltered.h5ad"),
    "--pct-counts-mt-max", cfg.PCT_COUNTS_MT_MAX,
    "--doublet-score-max", cfg.DOUBLET_SCORE_MAX,
    "--total-counts-limits", *cfg.TOTAL_COUNTS_LIMITS,
    "--n-genes-by-counts-limits", *cfg.N_GENES_BY_COUNTS_LIMITS,
    "--n-mads-lower", cfg.N_MADS_LOWER,
    "--n-mads-upper", cfg.N_MADS_UPPER,
    "--adata-output", cfg.out("filtered.h5ad"),
    "--output-removed-cells-file", cfg.out("filtered_out_cells.csv"),
)

## 3. Check the result

In [ ]:
import anndata as ad

filtered = ad.read_h5ad(cfg.out("filtered.h5ad"), backed="r")
kept = filtered.obs["sample"].value_counts().rename("kept")
total = obs["sample"].value_counts().rename("merged")
per_sample = pd.concat([total, kept], axis=1)
per_sample["kept %"] = (100 * per_sample["kept"] / per_sample["merged"]).round(1)
print(f"kept {filtered.n_obs:,} of {len(obs):,} cells")
per_sample

## 4. Compare with the curated release

Both runs record every removed cell and the filters it failed (workflow v5.0.0 and later).

In [ ]:
def failed_counts(path) -> pd.Series:
    removed = pd.read_csv(path, index_col=0)
    return pd.concat([pd.Series({"removed cells": len(removed)}), removed.filter(like="failed_").sum()])

counts = {"this run": failed_counts(cfg.out("filtered_out_cells.csv"))}
removed_csv = cfg.curated("filtered_out_cells.csv")
if removed_csv.exists():
    counts["release"] = failed_counts(removed_csv)
else:
    print(f"{removed_csv.name} not in the curated release")
pd.DataFrame(counts)

**Previous:** [02_merge_and_plot_qc](02_merge_and_plot_qc.ipynb) · **Next:** [04_mmc](04_mmc.ipynb)